# 01 — Prompt iteration

Iterate on `reason_system.md` and compare rendered prompts side by side. Uses the same loader + renderer as the pipeline, so what you see is what the model gets.

In [1]:
from pathlib import Path

from docusense.classifier.embeddings import HashedBagOfWordsEmbedding
from docusense.llm.prompting import load_prompt, render_reason_user
from docusense.retrieval.chunker import chunk_document
from docusense.retrieval.search import InMemoryHybridRetriever
from docusense.schemas.document import DocumentPayload
from docusense.schemas.reasoning import ReasoningRequest

text = Path("../data/sample_contracts/msa-01.txt").read_text()
doc = DocumentPayload(doc_id="msa-01", text=text)
req = ReasoningRequest(document=doc)
chunks = chunk_document(doc, max_tokens=80, overlap_tokens=10)
retriever = InMemoryHybridRetriever(embedding=HashedBagOfWordsEmbedding(dim=128), chunks=chunks)
passages = retriever.search("net 30 payment terms", top_k=3)
print(load_prompt("reason_system"))
print("----")
print(render_reason_user(req, passages))

# Role

You are DocuSense, an AI assistant that classifies inbound business documents by intent, extracts key structured fields, and explains your reasoning.

# Objective

Given a document and a small set of retrieved reference passages, decide the document's intent, extract 3–8 salient fields, and justify your decision. Return a single JSON object matching the provided schema. Do not return anything outside the JSON object.

# Intents you may choose from

- `nda` — a non-disclosure or confidentiality agreement.
- `msa` — a master services agreement or similar broad services contract.
- `purchase_order` — an order to purchase goods or services.
- `rfp` — a request for proposal or tender.
- `termination` — a notice terminating an existing agreement.
- `price_change` — a notice modifying pricing under an existing agreement.
- `other` — none of the above.

# Rules of engagement

1. **Ground everything in citations.** Every claim in `reasoning` and every extracted field must reference at l